# Coupled Wastewater Treatment and Carbon Mineralization Model
## Inspired by CREW's Process Intensification Technology

This notebook models the coupled system of:
1. **Biological wastewater treatment** — ASM1 (Henze et al. 1987)
2. **Carbon mineralization** — CO₂ sequestration via mineral alkalinity

**Run all cells top to bottom** to simulate the reactor and see interactive results.

> The same ODE framework as the AWL ship reactor model (Dong et al. 2025), applied to wastewater biogeochemistry.


## 1. Imports

In [ ]:
"""
Coupled Wastewater Treatment and Carbon Mineralization Model
Inspired by CREW's process intensification technology.

Models:
1. ASM1 (Activated Sludge Model No. 1, Henze et al. 1987) - biological treatment
2. CO2 mineral sequestration - carbon capture via alkalinity enhancement

Same mathematical framework as Dong et al. (2025) AWL reactor model.
"""

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'

## 2. Display Settings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

## 3. Model Parameters

Three parameter dictionaries define the full coupled model.

**ASM1 kinetics** (Henze et al. 1987, T=20°C): max growth rates, half-saturation constants, yield coefficients, and decay rates for heterotrophs (COD removers) and autotrophs (nitrifiers).

**Mineral parameters**: CO₂ mineralization rate constant, Henry's constant, atmospheric pCO₂ (equilibrium reference), and mineral loading (kg mineral/m³ reactor).

**Reactor parameters**: HRT = 12 hours, SRT = 10 days, aeration rate, and typical municipal wastewater influent concentrations.


In [ ]:
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
ASM1_PARAMS = {
    'mu_H':          6.0,    # Max specific growth rate, heterotrophs [1/d]
    'K_S':           20.0,   # Half-saturation, substrate [g COD/m3]
    'K_OH':          0.2,    # Half-saturation, O2 (heterotrophs) [g O2/m3]
    'K_NO':          0.5,    # Half-saturation, NO3 [g N/m3]
    'b_H':           0.62,   # Decay rate, heterotrophs [1/d]
    'eta_g':         0.8,    # Anoxic growth correction factor
    'Y_H':           0.67,   # Yield, heterotrophs [g COD biomass / g COD substrate]
    'mu_A':          0.8,    # Max specific growth rate, autotrophs [1/d]
    'K_NH':          1.0,    # Half-saturation, NH4 [g N/m3]
    'K_OA':          0.4,    # Half-saturation, O2 (autotrophs) [g O2/m3]
    'b_A':           0.05,   # Decay rate, autotrophs [1/d]
    'Y_A':           0.24,   # Yield, autotrophs [g COD / g N oxidized]
    'i_XB':          0.086,  # N content of biomass [g N / g COD]
    'f_P':           0.08,   # Fraction of biomass as endogenous residue
    'CO2_per_COD':   1.375,  # g CO2 produced per g COD oxidized
}

In [ ]:
MINERAL_PARAMS = {
    'k_min':           0.05,   # Mineralization rate constant [1/d per kg/m3 mineral]
    'KH_CO2':          3.4e-2, # Henry constant CO2 [mol/L/atm] at 20C
    'pCO2_atm':        420e-6, # Atmospheric pCO2 [atm]
    'mineral_loading': 10.0,   # kg mineral / m3 reactor
    'MW_CO2':          44.01,  # g/mol
}

## 4. Kinetic Functions

- **`monod(S, K)`**: Michaelis-Menten kinetics — S/(K+S), approaches 1 when S >> K
- **`switching(S, K)`**: Inhibition function — K/(K+S), near 1 when inhibitor is absent (used for anoxic switching when DO is low)
- **`co2_eq_g_m3`**: equilibrium dissolved CO₂ with atmosphere (~0.63 g/m³ at 420 ppm, 20°C)
- **`co2_mineralization_rate`**: first-order rate for CO₂ sequestration — proportional to mineral loading and excess CO₂ above atmospheric equilibrium
- **`asm1_rates`**: five ASM1 process rates (aerobic growth, denitrification, nitrification, two decay terms)


In [ ]:
REACTOR_PARAMS = {
    'HRT':     0.5,    # Hydraulic retention time [d] = 12 hours
    'SRT':     10.0,   # Sludge retention time [d]
    'Q_O2':    200.0,  # Aeration rate [g O2/m3/d]
    'SS_in':   200.0,  # Influent COD [g COD/m3]
    'SO_in':   0.0,    # Influent DO [g O2/m3]
    'SNH_in':  40.0,   # Influent ammonium [g N/m3]
    'SNO_in':  0.0,    # Influent nitrate [g N/m3]
    'SALK_in': 7.0,    # Influent alkalinity [mol HCO3/m3]
    'SCO2_in': 0.5,    # Influent dissolved CO2 [g CO2/m3]
    'XBH_in':  0.0,    # Influent heterotroph biomass
    'XBA_in':  0.0,    # Influent autotroph biomass
}

In [ ]:
def monod(S, K):
    return max(S, 0.0) / (K + max(S, 0.0))

In [ ]:
def switching(S, K):
    return K / (K + max(S, 0.0))

In [ ]:
def co2_eq_g_m3(mineral_params):
    """Equilibrium dissolved CO2 with atmosphere [g CO2/m3]"""
    CO2_eq_mol_L = mineral_params['KH_CO2'] * mineral_params['pCO2_atm']
    return CO2_eq_mol_L * mineral_params['MW_CO2'] * 1000.0

In [ ]:
def co2_mineralization_rate(S_CO2, mineral_loading, mp):
    """Rate of CO2 sequestration by mineral dissolution [g CO2/m3/d]"""
    CO2_eq = co2_eq_g_m3(mp)
    excess = max(S_CO2 - CO2_eq, 0.0)
    return mp['k_min'] * mineral_loading * excess

## 5. Coupled ODE System

Eight coupled mass balance ODEs — one per state variable — in a single CSTR:

| Variable | Description | Key processes |
|----------|-------------|---------------|
| S_S | COD (g/m³) | Consumed by aerobic and anoxic heterotrophic growth |
| S_O | Dissolved O₂ (g/m³) | Added by aeration; consumed by heterotrophs and nitrifiers |
| S_NH | Ammonium (g N/m³) | Consumed by nitrification and biomass synthesis |
| S_NO | Nitrate (g N/m³) | Produced by nitrification; consumed by denitrification |
| X_BH | Heterotroph biomass (g COD/m³) | Grows on COD; washes out at 1/SRT |
| X_BA | Autotroph biomass (g COD/m³) | Grows by nitrification; washes out at 1/SRT |
| S_ALK | Alkalinity (mol HCO₃⁻/m³) | **Increases from mineral CO₂ sequestration** |
| S_CO₂ | Dissolved CO₂ (g/m³) | Produced by respiration; **consumed by mineralization** |

The coupling appears in the last two equations: biogenic CO₂ drives mineral dissolution, which adds alkalinity (permanent CO₂ storage) and depletes dissolved CO₂.


In [ ]:
def asm1_rates(state, p):
    """Calculate ASM1 process rates. Returns (rho1, rho2, rho3, rho4, rho5)."""
    S_S, S_O, S_NH, S_NO, X_BH, X_BA = [max(v, 0.0) for v in state[:6]]

    rho1 = p['mu_H'] * monod(S_S, p['K_S']) * monod(S_O, p['K_OH']) * X_BH
    rho2 = (p['mu_H'] * monod(S_S, p['K_S']) * switching(S_O, p['K_OH'])
            * monod(S_NO, p['K_NO']) * p['eta_g'] * X_BH)
    rho3 = p['mu_A'] * monod(S_NH, p['K_NH']) * monod(S_O, p['K_OA']) * X_BA
    rho4 = p['b_H'] * X_BH
    rho5 = p['b_A'] * X_BA
    return rho1, rho2, rho3, rho4, rho5

## 6. Simulation Function

In [ ]:
def coupled_odes(t, y, rp, p, mp):
    S_S, S_O, S_NH, S_NO, X_BH, X_BA, S_ALK, S_CO2 = y

    inv_HRT = 1.0 / rp['HRT']
    inv_SRT = 1.0 / rp['SRT']

    rho1, rho2, rho3, rho4, rho5 = asm1_rates(y, p)

    # Dissolved CO2 at atmospheric equilibrium
    CO2_eq = co2_eq_g_m3(mp)
    k_La_CO2 = 5.0  # d-1, CO2 gas transfer coefficient

    # Mineralization rate
    r_min = co2_mineralization_rate(max(S_CO2, 0.0), mp['mineral_loading'], mp)

    # Alkalinity gain from mineralization: 2 mol HCO3 per mol CO2 mineralized
    r_alk_mineral = r_min * 2.0 / mp['MW_CO2']  # mol HCO3/m3/d

    # Mass balances
    dSS  = inv_HRT * (rp['SS_in']  - S_S)  - (1.0/p['Y_H']) * (rho1 + rho2)
    dSO  = inv_HRT * (rp['SO_in']  - S_O)  + rp['Q_O2'] \
           - ((1.0 - p['Y_H']) / p['Y_H']) * rho1 \
           - (4.57 - p['Y_A']) / p['Y_A'] * rho3
    dSNH = inv_HRT * (rp['SNH_in'] - S_NH) \
           - p['i_XB'] * (rho1 + rho2) \
           - (p['i_XB'] + 1.0/p['Y_A']) * rho3
    dSNO = inv_HRT * (rp['SNO_in'] - S_NO) \
           + (1.0/p['Y_A']) * rho3 \
           - ((1.0 - p['Y_H']) / (2.86 * p['Y_H'])) * rho2
    dXBH = inv_HRT * (rp['XBH_in'] - X_BH) - inv_SRT * X_BH + rho1 + rho2 - rho4
    dXBA = inv_HRT * (rp['XBA_in'] - X_BA) - inv_SRT * X_BA + rho3 - rho5
    dSALK = inv_HRT * (rp['SALK_in'] - S_ALK) \
            - 0.07 * rho3 / 14.0 \
            + 0.07 * rho2 / 14.0 \
            + r_alk_mineral
    dSCO2 = inv_HRT * (rp['SCO2_in'] - S_CO2) \
            + p['CO2_per_COD'] * (1.0 - p['Y_H']) * rho1 \
            + 0.15 * rho3 \
            - r_min \
            - k_La_CO2 * (max(S_CO2, 0.0) - CO2_eq)

    return [dSS, dSO, dSNH, dSNO, dXBH, dXBA, dSALK, dSCO2]

## 6. Simulation Function

`run_simulation` integrates the ODE system to steady state using LSODA. Default: 30 days, 3000 time points.

In [ ]:
def run_simulation(reactor_params, asm_params, mineral_params,
                   y0=None, t_end=30.0, n_points=3000):
    """Run coupled wastewater + carbon sequestration simulation to steady state."""
    rp = reactor_params.copy()
    rp.setdefault('XBH_in', 0.0)
    rp.setdefault('XBA_in', 0.0)

    if y0 is None:
        y0 = [rp['SS_in'], 2.0, rp['SNH_in'], 0.0,
              100.0, 10.0, rp['SALK_in'], 0.5]

    sol = solve_ivp(
        coupled_odes,
        (0, t_end), y0,
        args=(rp, asm_params, mineral_params),
        t_eval=np.linspace(0, t_end, n_points),
        method='LSODA', rtol=1e-6, atol=1e-9
    )
    return sol

## 7. Run Simulation — Default Parameters

Runs the coupled model for 30 days to reach steady state.  
**Expected results at default settings (HRT=12h, SRT=10d, mineral loading=10 kg/m³):**
- COD removal: ~90%
- NH₄⁺ removal: partial (nitrifiers need longer SRT for full removal)
- CO₂ sequestration: modest at default mineral loading — increase to see improvement


In [ ]:
sol = run_simulation(REACTOR_PARAMS, ASM1_PARAMS, MINERAL_PARAMS)
t = sol.t
S_S, S_O, S_NH, S_NO, X_BH, X_BA, S_ALK, S_CO2 = sol.y

import numpy as np
rho_arr = np.array([asm1_rates(sol.y[:, i], ASM1_PARAMS) for i in range(len(t))])
CO2_prod = ASM1_PARAMS['CO2_per_COD'] * (1 - ASM1_PARAMS['Y_H']) * rho_arr[:, 0]
CO2_min  = np.array([co2_mineralization_rate(max(S_CO2[i], 0.0),
                     MINERAL_PARAMS['mineral_loading'], MINERAL_PARAMS)
                     for i in range(len(t))])
seq_eff = np.where(CO2_prod > 0, CO2_min / CO2_prod * 100, 0.0)
CO2_eq_val = co2_eq_g_m3(MINERAL_PARAMS)

print(f"Steady-state results (day {t[-1]:.0f}):")
print(f"  COD removal:         {(1-S_S[-1]/REACTOR_PARAMS['SS_in'])*100:.1f}%")
print(f"  NH4+ removal:        {(1-S_NH[-1]/REACTOR_PARAMS['SNH_in'])*100:.1f}%")
print(f"  NO3- produced:       {S_NO[-1]:.1f} g N/m3")
print(f"  Dissolved O2:        {S_O[-1]:.2f} g/m3")
print(f"  Alkalinity gain:     {S_ALK[-1]-REACTOR_PARAMS['SALK_in']:.3f} mol HCO3/m3")
print(f"  Dissolved CO2:       {S_CO2[-1]:.2f} g/m3  (atm eq: {CO2_eq_val:.2f})")
print(f"  CO2 sequestration:   {seq_eff[-1]:.1f}% of biogenic CO2 mineralized")


## 8. Interactive Treatment Performance Dashboard (Plotly)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=3,
    subplot_titles=[
        "COD Removal", "Ammonium (NH4+)", "Nitrate (NO3-)",
        "Dissolved Oxygen", "Heterotroph Biomass", "Nitrifier Biomass",
        "Alkalinity", "Dissolved CO2", "Carbon Sequestration"
    ], vertical_spacing=0.12, horizontal_spacing=0.1)

def hline(y, color):
    return go.Scatter(x=[t[0], t[-1]], y=[y, y], mode='lines',
                      line=dict(dash='dash', color=color, width=1), showlegend=False)

# Row 1
fig.add_trace(go.Scatter(x=t, y=S_S, name='COD', line=dict(color='#1f77b4', width=2)), row=1, col=1)
fig.add_trace(hline(REACTOR_PARAMS['SS_in'], '#1f77b4'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=S_NH, name='NH4+', line=dict(color='orange', width=2)), row=1, col=2)
fig.add_trace(hline(REACTOR_PARAMS['SNH_in'], 'orange'), row=1, col=2)
fig.add_trace(go.Scatter(x=t, y=S_NO, name='NO3-', line=dict(color='red', width=2)), row=1, col=3)

# Row 2
fig.add_trace(go.Scatter(x=t, y=S_O, name='DO', line=dict(color='cyan', width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=X_BH, name='Heterotrophs', line=dict(color='green', width=2)), row=2, col=2)
fig.add_trace(go.Scatter(x=t, y=X_BA, name='Nitrifiers', line=dict(color='purple', width=2)), row=2, col=3)

# Row 3
fig.add_trace(go.Scatter(x=t, y=S_ALK, name='Alkalinity', line=dict(color='mediumpurple', width=2)), row=3, col=1)
fig.add_trace(hline(REACTOR_PARAMS['SALK_in'], 'mediumpurple'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=S_CO2, name='CO2', line=dict(color='brown', width=2)), row=3, col=2)
fig.add_trace(hline(CO2_eq_val, 'brown'), row=3, col=2)
fig.add_trace(go.Scatter(x=t, y=CO2_prod, name='CO2 produced', line=dict(color='red', width=2)), row=3, col=3)
fig.add_trace(go.Scatter(x=t, y=CO2_min, name='CO2 mineralized', line=dict(color='green', width=2)), row=3, col=3)

for r, c, label in [(1,1,'g COD/m3'),(1,2,'g N/m3'),(1,3,'g N/m3'),
                     (2,1,'g O2/m3'),(2,2,'g COD/m3'),(2,3,'g COD/m3'),
                     (3,1,'mol HCO3/m3'),(3,2,'g CO2/m3'),(3,3,'g CO2/m3/d')]:
    fig.update_yaxes(title_text=label, row=r, col=c)
    fig.update_xaxes(title_text='Time (days)', row=3, col=c)

fig.update_layout(height=800,
    title='ASM1 + Carbon Mineralization — Treatment Performance',
    hovermode='x unified', legend=dict(orientation='h', y=-0.05))
fig.show()


## 9. Carbon Sequestration Analysis (Plotly)

In [ ]:
CO2_stripping = np.maximum(S_CO2 - CO2_eq_val, 0.0) * 5.0

fig2 = make_subplots(rows=1, cols=2,
    subplot_titles=["CO2 Fate Over Time", "Sequestration Efficiency (%)"],
    column_widths=[0.6, 0.4])

fig2.add_trace(go.Scatter(x=t, y=CO2_prod, name='CO2 produced',
    fill='tozeroy', line=dict(color='rgba(200,0,0,0.8)', width=2),
    fillcolor='rgba(200,0,0,0.12)'), row=1, col=1)
fig2.add_trace(go.Scatter(x=t, y=CO2_min, name='CO2 mineralized',
    fill='tozeroy', line=dict(color='rgba(0,150,0,0.8)', width=2),
    fillcolor='rgba(0,150,0,0.25)'), row=1, col=1)
fig2.add_trace(go.Scatter(x=t, y=CO2_stripping, name='Stripped to atmosphere',
    fill='tozeroy', line=dict(color='rgba(100,100,200,0.6)', width=1, dash='dot'),
    fillcolor='rgba(100,100,200,0.08)'), row=1, col=1)
fig2.add_trace(go.Scatter(x=t, y=seq_eff, name='Sequestration efficiency',
    line=dict(color='darkgreen', width=3),
    fill='tozeroy', fillcolor='rgba(0,150,0,0.1)'), row=1, col=2)

fig2.update_xaxes(title_text='Time (days)')
fig2.update_yaxes(title_text='g CO2/m3/d', row=1, col=1)
fig2.update_yaxes(title_text='%', row=1, col=2, range=[0, 100])
fig2.update_layout(height=430, title='Carbon Sequestration Analysis',
    hovermode='x unified', legend=dict(orientation='h', y=-0.15))
fig2.show()


## 10. Sensitivity to Mineral Loading (Plotly)

Increasing mineral loading raises CO₂ sequestration efficiency.  
Try adjusting `mineral_loadings` range to explore.


In [ ]:
mineral_loadings = np.linspace(0, 50, 20)
seq_effs_s, alk_gains_s, cod_removals_s = [], [], []

for ml in mineral_loadings:
    mp_t = MINERAL_PARAMS.copy(); mp_t['mineral_loading'] = ml
    sol_t = run_simulation(REACTOR_PARAMS, ASM1_PARAMS, mp_t, t_end=40.0, n_points=400)
    if sol_t.success:
        SS_ss, SCO2_ss, SALK_ss = sol_t.y[0,-1], sol_t.y[7,-1], sol_t.y[6,-1]
        rho_ss = asm1_rates(sol_t.y[:,-1], ASM1_PARAMS)
        CO2_p = ASM1_PARAMS['CO2_per_COD']*(1-ASM1_PARAMS['Y_H'])*rho_ss[0]
        r_min = co2_mineralization_rate(max(SCO2_ss,0), ml, mp_t)
        seq_effs_s.append(r_min/CO2_p*100 if CO2_p>0 else 0)
        alk_gains_s.append(SALK_ss - REACTOR_PARAMS['SALK_in'])
        cod_removals_s.append((1-SS_ss/REACTOR_PARAMS['SS_in'])*100)
    else:
        seq_effs_s.append(None); alk_gains_s.append(None); cod_removals_s.append(None)

fig3 = make_subplots(rows=1, cols=3,
    subplot_titles=['CO2 Sequestration Efficiency (%)',
                    'Alkalinity Gain (mol HCO3/m3)', 'COD Removal (%)'])

for col, (data, color, ylabel) in enumerate(zip(
    [seq_effs_s, alk_gains_s, cod_removals_s],
    ['green','purple','blue'],
    ['%','mol HCO3/m3','%']), start=1):
    fig3.add_trace(go.Scatter(x=mineral_loadings, y=data,
        mode='lines+markers', showlegend=False,
        line=dict(color=color, width=2), marker=dict(size=6)), row=1, col=col)
    fig3.update_xaxes(title_text='Mineral loading (kg/m3)', row=1, col=col)
    fig3.update_yaxes(title_text=ylabel, row=1, col=col)

fig3.update_layout(height=400, title='Sensitivity to Mineral Loading',
    hovermode='x unified')
fig3.show()


## 11. Sensitivity to HRT and SRT (Plotly)

HRT controls residence time for substrate removal.  
SRT controls nitrifier retention — longer SRT enables better NH₄⁺ removal.


In [ ]:
HRTs = np.linspace(0.2, 1.0, 8)
SRTs = [5, 10, 20]
colors_srt = ['#1f77b4','#ff7f0e','#2ca02c']

fig4 = make_subplots(rows=1, cols=3,
    subplot_titles=['COD Removal (%)','NH4+ Removal (%)','CO2 Sequestration (%)'])

for j, srt in enumerate(SRTs):
    cod_r, n_r, seq_r = [], [], []
    for hrt in HRTs:
        rp_t = REACTOR_PARAMS.copy(); rp_t['HRT']=hrt; rp_t['SRT']=srt
        sol_t = run_simulation(rp_t, ASM1_PARAMS, MINERAL_PARAMS, t_end=50.0, n_points=400)
        if sol_t.success:
            cod_r.append((1-sol_t.y[0,-1]/REACTOR_PARAMS['SS_in'])*100)
            n_r.append((1-sol_t.y[2,-1]/REACTOR_PARAMS['SNH_in'])*100)
            rho_ss = asm1_rates(sol_t.y[:,-1], ASM1_PARAMS)
            CO2_p = ASM1_PARAMS['CO2_per_COD']*(1-ASM1_PARAMS['Y_H'])*rho_ss[0]
            r_min = co2_mineralization_rate(max(sol_t.y[7,-1],0),
                                             MINERAL_PARAMS['mineral_loading'], MINERAL_PARAMS)
            seq_r.append(r_min/CO2_p*100 if CO2_p>0 else 0)
        else:
            cod_r.append(None); n_r.append(None); seq_r.append(None)

    label = f'SRT={srt}d'
    hrt_h = HRTs*24
    for col, data in enumerate([cod_r, n_r, seq_r], start=1):
        fig4.add_trace(go.Scatter(x=hrt_h, y=data, name=label,
            mode='lines+markers', line=dict(color=colors_srt[j], width=2),
            marker=dict(size=5), showlegend=(col==1)), row=1, col=col)
        fig4.update_xaxes(title_text='HRT (hours)', row=1, col=col)
        fig4.update_yaxes(title_text='%', row=1, col=col, range=[0,105])

fig4.update_layout(height=420, title='Sensitivity to HRT and SRT',
    hovermode='x unified', legend=dict(orientation='h', y=-0.15))
fig4.show()
